This notebook explores variability in hail's python (macro)-benchmarks when
said benchmarks are executed on the hail batch service. The analyses within 
are based off the methods proposed in [1], albeit slightly modified for long
running benchmarks. The goals of these analyses are

- to determine if we can detect slowdowns of 5% or less reliably when running
  benchmarks on hail batch.
- to identify configurations (number of batch jobs x iterations) that allow us
  to detect slowdowns efficiently (ie without excesssive time and money).

[1] Laaber et al., Software Microbenchmarking in the Cloud.How Bad is it Really?
    https://dl.acm.org/doi/10.1007/s10664-019-09681-1

In [ ]:
from pathlib import Path
from typing import Dict, List

from benchmark.tools import maybe
from benchmark.tools.impex import dump_tsv, import_timings
from benchmark.tools.plotting import plot_mean_time_per_instance, plot_trial_against_time
from benchmark.tools.statistics import (
    bootstrap_mean_confidence_interval,
    laaber_mds,
    schultz_mds,
    variability,
)
from IPython.display import clear_output
from plotly.io import renderers

import hail as hl

renderers.default = 'notebook_connected'

In [ ]:
hl.init(backend='spark', idempotent=True, local_tmpdir='/tmp/mds')
hl._set_flags(use_new_shuffle='1', lower='1')

In [ ]:
# Import benchmark data
# ---------------------
#
# benchmarks under `hail/python/benchmarks` are executed with a custom pytest
# plugin and their results are output as json lines (.jsonl).
# Unscrupulously, we use hail to analyse itself.

with hl.TemporaryFilename(dir='/tmp') as tsvfile:
    timings = Path(tsvfile)
    dump_tsv(Path('in/benchmarks.jsonl'), timings)
    ht = import_timings(timings)
    ht = ht.checkpoint('out/imported.ht', overwrite=True)

In [ ]:
ht = hl.read_table('out/imported.ht')

benchmarks = ht.aggregate(hl.agg.collect_as_set(ht.name))
print(*benchmarks, sep='\n')

In [ ]:
# In this next section, we'll estimate the number of iterations required for
# a benchmark to reach a steady-state, or the number of so-called "burn-in"
# iterations.


def filter_burn_in_iterations(ht: hl.Table, first_stable_index: Dict[str, int]) -> hl.Table:
    ht = ht.annotate_globals(first_stable_index=first_stable_index)
    return ht.select(
        instances=ht.instances.map(
            lambda instance: instance.annotate(
                trials=hl.filter(
                    lambda t: t.iteration >= ht.first_stable_index.get(ht.name, 0),
                    instance.trials,
                ),
            )
        ),
    )


first_stable_index = {
    'benchmark_matrix_table_scan_count_cols_2': 20,
    'benchmark_kyle_sex_specific_qc': 7,
    'benchmark_matrix_table_entries_table_no_key': 10,
    'benchmark_test_left_join_region_memory': 10,
    'benchmark_matrix_table_call_stats_star_star': 8,
    'benchmark_ndarray_matmul_int64': 10,
    'benchmark_table_scan_prev_non_null': 20,
    'benchmark_export_range_matrix_table_row_p100': 8,
    'benchmark_join_partitions_table[1000-10]': 12,
    'benchmark_split_multi': 4,
    'benchmark_matrix_table_filter_entries': 6,
    'benchmark_mt_localize_and_collect': 5,
    'benchmark_table_range_array_range_force_count': 6,
    'benchmark_variant_qc': 8,
    'benchmark_matrix_table_take_entry': 10,
    'benchmark_union_partitions_table[1000-10]': 12,
    'benchmark_import_bgen_info_score': 12,
    'benchmark_export_range_matrix_table_col_p100': 15,
    'benchmark_join_partitions_table[100-100]': 10,
    'benchmark_logistic_regression_rows_wald': 5,
    'benchmark_matrix_table_decode_and_count_just_gt': 5,
    'benchmark_table_range_force_count': 10,
    'benchmark_matrix_table_array_arithmetic': 20,
    'benchmark_sentinel_read_gunzip': 10,
    'benchmark_join_partitions_table[100-10]': 10,
    'benchmark_matrix_table_take_col': 10,
    'benchmark_join_partitions_table[1000-1000]': 10,
    'benchmark_read_force_count_partitions[10]': 10,
    'benchmark_import_and_transform_gvcf': 10,
    'benchmark_table_take': 25,
    'benchmark_write_range_table[10000000-100]': 3,
    'benchmark_table_read_force_count_strings': 10,
    'benchmark_test_inner_join_region_memory': 10,
    'benchmark_write_range_table[10000000-1000]': 6,
    'benchmark_join_partitions_table[100-1000]': 10,
    'benchmark_matrix_table_cols_show': 16,
    'benchmark_python_only_10k_transform': 10,
    'benchmark_union_partitions_table[100-100]': 5,
    'benchmark_read_force_count_partitions[1000]': 10,
    'benchmark_sample_qc': 10,
    'benchmark_matrix_table_entries_show': 15,
    'benchmark_sentinel_cpu_hash_1': 5,
    'benchmark_analyze_benchmarks': 10,
    'benchmark_import_gvcf_force_count': 8,
    'benchmark_table_range_join_1b_1b': 10,
    'benchmark_import_bgen_filter_count': 18,
    'benchmark_table_read_force_count_ints': 5,
    'benchmark_split_multi_hts': 10,
    'benchmark_union_partitions_table[100-10]': 20,
    'benchmark_table_range_join_1b_1k': 20,
    'benchmark_matrix_table_aggregate_entries': 8,
    'benchmark_export_vcf': 15,
    'benchmark_matrix_table_many_aggs_col_wise': 10,
    'benchmark_test_map_filter_region_memory': 15,
    'benchmark_import_vcf_count_rows': 1,
    'benchmark_ndarray_matmul_float64': 6,
    'benchmark_table_python_construction': 10,
    'benchmark_per_row_stats_star_star': 10,
    'benchmark_matrix_table_take_row': 10,
    'benchmark_import_bgen_force_count_just_gp': 20,
    'benchmark_matrix_table_entries_table': 10,
    'benchmark_union_partitions_table[10-100]': 5,
    'benchmark_union_partitions_table[10-10]': 5,
    'benchmark_table_aggregate_array_sum': 5,
    'benchmark_mt_group_by_memory_usage': 5,
    'benchmark_shuffle_key_by_aggregate_good_locality': 5,
    'benchmark_hwe_normalized_pca_blanczos_small_data_0_iterations': 5,
    'benchmark_read_force_count_partitions[100]': 8,
    'benchmark_table_show': 20,
    'benchmark_concordance': 3,
    'benchmark_matrix_table_filter_entries_unfilter': 3,
    'benchmark_union_partitions_table[10-1000]': 8,
    'benchmark_matrix_table_many_aggs_row_wise': 4,
    'benchmark_sum_table_of_ndarrays': 10,
    'benchmark_python_only_10k_combine': 20,
    'benchmark_union_partitions_table[100-1000]': 15,
    'benchmark_join_partitions_table[1000-100]': 10,
    'benchmark_union_partitions_table[1000-100]': 6,
    'benchmark_shuffle_key_by_aggregate_bad_locality': 8,
    'benchmark_vds_combiner_chr22': 10,
    'benchmark_write_range_table[10000000-10]': 5,
    'benchmark_read_with_index[1000]': 8,
    'benchmark_genetics_pipeline': 10,
    'benchmark_import_vcf_write': 5,
    'benchmark_import_bgen_force_count_all': 20,
    'benchmark_linear_regression_rows': 10,
    'benchmark_export_range_matrix_table_entry_field_p100': 3,
    'benchmark_blockmatrix_write_from_entry_expr_range_mt': 10,
    'benchmark_matrix_table_rows_show': 15,
    'benchmark_matrix_table_show': 15,
    'benchmark_write_range_matrix_table_p100': 8,
    'benchmark_test_head_and_tail_region_memory': 10,
    'benchmark_matrix_table_rows_is_transition': 10,
    'benchmark_matrix_table_decode_and_count': 8,
    'benchmark_matrix_table_rows_force_count': 30,
    'benchmark_ndarray_addition': 10,
    'benchmark_table_range_means': 10,
    'benchmark_write_profile_mt': 15,
    'benchmark_make_ndarray': 5,
    'benchmark_matrix_table_scan_count_rows_2': 5,
    'benchmark_read_decode_gnomad_coverage': 10,
    'benchmark_table_aggregate_approx_cdf': 16,
    'benchmark_table_scan_sum_1k_partitions': 6,
    'benchmark_union_partitions_table[1000-1000]': 10,
    'benchmark_variant_and_sample_qc_nested_with_filters_4_counts': 20,
    'benchmark_group_by_take_rekey': 10,
    'benchmark_table_annotate_many_flat': 18,
    'benchmark_table_import_strings': 3,
    'benchmark_table_aggregate_int_stats': 18,
    'benchmark_variant_and_sample_qc': 18,
    'benchmark_table_foreign_key_join[1000000-1000]': 8,
    'benchmark_table_group_by_aggregate_sorted': 10,
    'benchmark_shuffle_key_rows_by_4096_byte_rows': 4,
    'benchmark_hwe_normalized_pca': 4,
    'benchmark_table_aggregate_downsample_dense': 4,
    'benchmark_join_partitions_table[10-1000]': 14,
    'benchmark_join_partitions_table[10-10]': 6,
    'benchmark_table_group_by_aggregate_unsorted': 7,
    'benchmark_shuffle_order_by_10m_int': 22,
    'benchmark_table_aggregate_take_by_strings': 10,
    'benchmark_join_partitions_table[10-100]': 4,
    'benchmark_variant_and_sample_qc_nested_with_filters_4': 20,
    'benchmark_table_aggregate_counter': 20,
    'benchmark_table_key_by_shuffle': 6,
    'benchmark_table_expr_take': 20,
    'benchmark_shuffle_key_rows_by_65k_byte_rows': 4,
    'benchmark_table_foreign_key_join[1000000-1000000]': 20,
    'benchmark_table_import_ints_impute': 10,
    'benchmark_group_by_collect_per_row': 8,
    'benchmark_table_aggregate_downsample_worst_case': 10,
    'benchmark_table_big_aggregate_compilation': 5,
    'benchmark_shuffle_key_rows_by_mt': 10,
    'benchmark_table_big_aggregate_compile_and_execute': 5,
    'benchmark_table_import_ints': 9,
    'benchmark_variant_and_sample_qc_nested_with_filters_2': 13,
    'benchmark_table_aggregate_linreg': 8,
}

In [ ]:
# Short of an accurate algorithm for computing this, you, noble reader, are
# tasked with the mind-numbing task of looking at graphs and picking numbers.
# This is an iterative process and you'll likely lose the will to live mid-way.
#
# Persevere, friend. Your sacrifice will not go unrewarded.
#
# In what follows, you'll be shown two plots. On the top will be the unfiltered
# benchmark times vs iteration for all batch jobs. The plot below will show the
# same benchmark filtered to the number of burn in iterations you selected
# previously.
#
# You'll be promted to enter a new first stable index for each benchmark until
# you arrive at a fixed point. Note that some benchmarks never really reach
# stability. In this case try to pick a value that compromises between cost and
# accuracy (ie if a benchmark is really slow, we don't want to be doing tons
# of burn in iterations, whereas for a fast benchmark we could justify more).
#
# Good luck.

names: List[str] = benchmarks  # type: ignore

while len(names) != 0:
    ft = filter_burn_in_iterations(ht, first_stable_index)
    __new_names, names = names, []
    for before, after in zip(plot_trial_against_time(ht, __new_names), plot_trial_against_time(ft, __new_names)):
        clear_output(wait=True)

        name: str = before.labels.title  # type: ignore
        cur_index = first_stable_index.get(name)

        after.labels.title = f'{name} (filtered iteration={cur_index})'
        before.show()
        after.show()

        new_index = maybe(int, input('Enter the first stable index (or blank keep same)') or None)

        if new_index is not None and new_index != cur_index:
            first_stable_index[name] = new_index
            names.append(name)

first_stable_index

In [ ]:
# As a final step of cleaning, we'll filter out trials that differ by some
# multiplier of the median for each instance


def filter_outliers(ht: hl.Table, factor: hl.Float64Expression) -> hl.Table:
    return ht.select(
        instances=ht.instances.map(
            lambda instance: instance.annotate(
                trials=hl.bind(
                    lambda median: instance.trials.filter(
                        lambda t: hl.max([t.time, median]) / hl.min([t.time, median]) < factor
                    ),
                    hl.median(instance.trials.map(lambda t: t.time)),
                )
            ),
        ),
    )


def filter_names(ht: hl.Table, names: hl.SetExpression) -> hl.Table:
    return ht.filter(names.contains(ht.name))


def filter_failed_trials(ht: hl.Table) -> hl.Table:
    return ht.annotate(
        instances=ht.instances.map(
            lambda i: i.annotate(
                trials=hl.filter(
                    lambda t: (~t.timed_out) | hl.is_missing(t.failure),
                    i.trials,
                ),
            )
        ),
    )


def filter_non_viable_instances(ht: hl.Table, ninstances: hl.Int32Expression, ntrials: hl.Int32Expression) -> hl.Table:
    ht = ht.select(
        instances=hl.filter(
            lambda instance: hl.len(instance.trials) >= ntrials,
            ht.instances,
        ),
    )

    return ht.filter(hl.len(ht.instances) >= ninstances)


ht = hl.read_table('out/imported.ht')
all: List[str] = ht.aggregate(hl.agg.collect_as_set(ht.name))  # type: ignore


ht = filter_names(ht, hl.set([n for n in all if n in first_stable_index]))
ht = filter_burn_in_iterations(ht, first_stable_index)
ht = filter_failed_trials(ht)
ht = filter_outliers(ht, hl.float64(10))
ht = filter_non_viable_instances(ht, 50, 50)
ht = ht.checkpoint('out/filtered.ht', overwrite=True)

benchmarks = ht.aggregate(hl.agg.collect_as_set(ht.name))

print('Filtered:', *(n for n in all if n not in set(benchmarks)), sep='\n')

In [ ]:
# These plots show the mean time per instance. This provides a visual way of
# identifying differences in instance type if there are multiple distinct layers

for f in plot_mean_time_per_instance(ht):
    f.show()

In [ ]:
# laaber et al. section 4

ht = ht.select(instances=ht.instances.trials.time)
variability(ht).show(len(benchmarks))

In [ ]:
# laaber et al. section 5 - boostrapping confidence intervals of the mean

bootstrap_mean_confidence_interval(ht, 1000, 0.95).show()

In [ ]:
# Laaber et al - Minimal-Detectable Slowdown

ht = hl.read_table('out/filtered.ht')
laaber_mds(ht).write('out/laaber-mds.ht', overwrite=True)
schultz_mds(ht).write('out/schultz-mds.ht', overwrite=True)

In [ ]:
benchmarks = hl.read_table('out/filtered.ht')
benchmarks = benchmarks.aggregate(hl.agg.collect_as_set((benchmarks.path, benchmarks.name)))
benchmarks = sorted(benchmarks)

mds, schultz = [hl.read_table(f'out/{m}-mds.ht') for m in ('laaber', 'schultz')]
mds = mds.annotate_globals(first_stable_index=first_stable_index)
mds = mds.select(nburn_in=mds.first_stable_index[mds.name], laaber=mds.row_value, schultz=schultz[mds.key])

benchmarks = [
    ('benchmark/hail/benchmark_table.py', 'benchmark_table_foreign_key_join[1000000-1000000]'),
    ('benchmark/hail/benchmark_table.py', 'benchmark_write_range_table[10000000-1000]'),
]

for path, name in benchmarks:
    clear_output(wait=True)
    print(f'{path}::{name}')

    t = mds.filter((mds.path == path) & (mds.name == name))
    t.filter(t.slowdown == 1).show(100_000)

    # Select minimum `(instances, trails)` that reduces the rate of false positives
    # to some acceptable value.
    config = input('instances,trials')
    if not config:
        continue

    clear_output(wait=True)
    print(f'{path}::{name}')

    # Now identify a configuration of instances, trials and burn-in iterations
    m, n = [int(x) for x in config.split(',')]
    t.filter((t.slowdown > 1) & (hl.tuple([t.ninstances, t.ntrials]) >= (m, n))).show(100_000)
    input('Press any key to continue')

mds.show()